# Gemma 4 E4B · Colab 대화
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm/blob/main/gemma4_colab.ipynb)

Google 공식 **Gemma 4 E4B QAT Q4_0 GGUF**를 GPU에서 실행하고 Gradio로 대화합니다.
텍스트 모델 파일은 약 **5.15GB**입니다. T4 16GB 기준 문맥 4,096토큰, 동시 생성 1개로 시작합니다.

1. **런타임 → 런타임 유형 변경 → T4 GPU**를 선택합니다.
2. 아래 셀을 위에서부터 실행합니다. 처음 CUDA 빌드와 다운로드에는 시간이 걸립니다.
3. 마지막 대화창 셀에서 로그인 비밀번호를 입력하고 표시되는 Gradio 링크를 엽니다.

무료 Colab은 GPU와 실행 시간을 보장하지 않습니다. 또한 노트북 UI를 벗어나 웹 UI를 주로 사용하는 방식은 제한되며 세션이 종료될 수 있습니다. 이 노트북은 해당 제한을 해제하지 않습니다.
[Colab 정책](https://research.google.com/colaboratory/faq.html) · [모델](https://huggingface.co/google/gemma-4-E4B-it-qat-q4_0-gguf)


## 1. 실행 설정
처음에는 기본값을 사용하세요. 모델 파일은 Colab 임시 디스크에 저장됩니다. 세션이 초기화되면 다시 다운로드합니다.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

MODEL_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"
MODEL_REVISION = "4b4a2c1d584be7264f87aac328a1bc739ce81b6c"
MODEL_FILENAME = "gemma-4-E4B_q4_0-it.gguf"
LLAMA_VERSION = "v0.6.0"
UV_VERSION = "0.12.23"
PYTHON_PACKAGES = ["gradio==6.29.1", "huggingface-hub==2.1.1", "requests==2.34.2"]

WORK_DIR = Path("/content/gemma4_chat")
CONTEXT_SIZE = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 1.0
TOP_P = 0.95
TOP_K = 64
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8080
GRADIO_PORT = 7860
CHAT_USERNAME = "gemma"
SYSTEM_PROMPT = "사용자와 한국어로 대화하세요. 질문에 명확하고 자연스럽게 답하세요."
STARTUP_TIMEOUT = 300
REQUEST_TIMEOUT = 300
LOG_INTERVAL = 15

WORK_DIR.mkdir(parents=True, exist_ok=True)
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
    check=True, capture_output=True, text=True,
)
print(gpu.stdout.strip())
GPU_ARCH = gpu.stdout.strip().splitlines()[0].split(",")[-1].strip().replace(".", "")
SERVER_URL = f"http://{SERVER_HOST}:{SERVER_PORT}"
print(f"문맥 {CONTEXT_SIZE}토큰 / 답변 최대 {MAX_NEW_TOKENS}토큰")


## 2. 실행 도구 설치와 CUDA 빌드
현재 배정된 GPU용으로 llama.cpp를 빌드합니다. 설치와 빌드는 Colab 런타임 안에서 진행됩니다.
설치·빌드 로그를 실시간으로 표시합니다. 출력이 없는 동안에는 15초마다 경과 시간을 표시합니다.


In [ ]:
import threading
import time
import urllib.request

def run_logged(command, **kwargs):
    started = time.monotonic()
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1, **kwargs,
    )

    def stream_output():
        with process.stdout:
            for line in process.stdout:
                print(line, end="", flush=True)

    reader = threading.Thread(target=stream_output, daemon=True)
    reader.start()
    try:
        while True:
            try:
                returncode = process.wait(timeout=LOG_INTERVAL)
                break
            except subprocess.TimeoutExpired:
                elapsed = int(time.monotonic() - started)
                print(f"[진행 중 · {elapsed}초 경과]", flush=True)
    except KeyboardInterrupt:
        process.terminate()
        raise
    reader.join()
    if returncode:
        raise subprocess.CalledProcessError(returncode, command)
    print(f"[단계 완료 · {time.monotonic() - started:.1f}초]", flush=True)

uv_dir = WORK_DIR / "uv"
uv_binary = uv_dir / "uv"
if not uv_binary.exists():
    print("1/3: Python 패키지 설치 도구 uv 다운로드", flush=True)
    installer_url = f"https://github.com/astral-sh/uv/releases/download/{UV_VERSION}/uv-installer.sh"
    with urllib.request.urlopen(installer_url) as response:
        installer = response.read()
    installer_path = WORK_DIR / "uv-installer.sh"
    installer_path.write_bytes(installer)
    install_env = os.environ.copy()
    install_env.update(UV_INSTALL_DIR=str(uv_dir), UV_NO_MODIFY_PATH="1")
    run_logged(["sh", str(installer_path)], env=install_env)

print("2/3: Gradio 등 Python 패키지 설치", flush=True)
run_logged(
    [str(uv_binary), "pip", "install", "--system", "--python", sys.executable, *PYTHON_PACKAGES],
)
print("빌드에 필요한 시스템 패키지 설치", flush=True)
run_logged(["apt-get", "update"])
run_logged(
    ["apt-get", "install", "-y", "git", "cmake", "build-essential", "libcurl4-openssl-dev"],
)

print("3/3: llama.cpp 소스 다운로드 및 CUDA 빌드", flush=True)
llama_dir = WORK_DIR / f"llama.cpp-{LLAMA_VERSION}"
if not llama_dir.exists():
    run_logged(
        ["git", "clone", "--progress", "--depth", "1", "--branch", LLAMA_VERSION,
         "https://github.com/ggml-org/llama.cpp.git", str(llama_dir)],
    )
build_dir = llama_dir / "build"
print("CUDA 빌드 설정", flush=True)
run_logged(
    ["cmake", "-S", str(llama_dir), "-B", str(build_dir),
     "-DGGML_CUDA=ON", f"-DCMAKE_CUDA_ARCHITECTURES={GPU_ARCH}",
     "-DCMAKE_BUILD_TYPE=Release"],
)
print("CUDA 컴파일 시작 — 아래에 빌드 퍼센트와 파일명이 표시됩니다.", flush=True)
run_logged(
    ["cmake", "--build", str(build_dir), "--config", "Release",
     "--target", "llama-server", "-j", "2"],
)
SERVER_BINARY = build_dir / "bin" / "llama-server"
print("CUDA 서버 빌드 완료", flush=True)


## 3. 모델 다운로드
Google 공식 양자화 파일만 다운로드합니다. 이미지·음성 입력용 파일은 이 텍스트 대화에서 사용하지 않습니다.


In [ ]:
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import enable_progress_bars
from tqdm.auto import tqdm

enable_progress_bars()
print(f"모델 다운로드 시작: {MODEL_FILENAME} (약 5.15GB)", flush=True)
print("다운로드 진행률·전송 속도·남은 시간은 아래 진행 막대에 표시됩니다.", flush=True)
download_started = time.monotonic()
model_path = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILENAME,
    revision=MODEL_REVISION,
    local_dir=WORK_DIR / "models",
    tqdm_class=tqdm,
)
print(f"다운로드 완료: {Path(model_path).stat().st_size / 1e9:.2f}GB "
      f"/ {time.monotonic() - download_started:.1f}초", flush=True)


## 4. 모델 실행
서버는 Colab 내부 주소에만 연결됩니다. 모델을 불러오는 동안 기다려 주세요. 실패하면 표시되는 로그에서 원인을 확인할 수 있습니다.


In [ ]:
import json
import time
import requests

if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=30)

log_path = WORK_DIR / "llama-server.log"
server_command = [
    str(SERVER_BINARY), "--model", model_path,
    "--host", SERVER_HOST, "--port", str(SERVER_PORT),
    "--ctx-size", str(CONTEXT_SIZE), "--parallel", "1",
    "--n-gpu-layers", "99", "--batch-size", "256", "--ubatch-size", "128",
    "--jinja", "--chat-template-kwargs", json.dumps({"enable_thinking": False}),
    "--reasoning-format", "deepseek",
]
with log_path.open("w") as log_file:
    server_process = subprocess.Popen(
        server_command, stdout=log_file, stderr=subprocess.STDOUT,
    )

print("모델 로딩 시작 — GPU와 메모리 할당 로그를 표시합니다.", flush=True)
loading_started = time.monotonic()
deadline = loading_started + STARTUP_TIMEOUT
last_progress = loading_started
with log_path.open(errors="replace") as live_log:
    while time.monotonic() < deadline:
        new_logs = live_log.read()
        if new_logs:
            print(new_logs, end="", flush=True)
        if server_process.poll() is not None:
            print(live_log.read(), end="", flush=True)
            raise RuntimeError("모델 실행에 실패했습니다. 위 서버 로그를 확인하세요.")
        try:
            if requests.get(f"{SERVER_URL}/health", timeout=2).status_code == 200:
                print(live_log.read(), end="", flush=True)
                print(f"모델 준비 완료 · {time.monotonic() - loading_started:.1f}초 "
                      "— 아래 대화창 셀을 실행하세요.", flush=True)
                break
        except requests.RequestException:
            pass
        if time.monotonic() - last_progress >= LOG_INTERVAL:
            print(f"[모델 로딩 중 · {int(time.monotonic() - loading_started)}초 경과]", flush=True)
            last_progress = time.monotonic()
        time.sleep(2)
    else:
        server_process.terminate()
        server_process.wait(timeout=30)
        print(live_log.read(), end="", flush=True)
        raise TimeoutError("모델 준비 시간이 초과되었습니다. 서버 로그를 확인하세요.")


## 5. Gradio 대화창
비밀번호는 숨겨진 입력으로 받습니다. 사용자 이름은 **gemma**입니다. 표시되는 `gradio.live` 링크를 열어 로그인하세요.

문맥 한도를 넘으면 대화창에서 대화를 지우고 새로 시작하세요. 대화 기록은 서버 파일에 저장하지 않습니다. 공유 링크는 외부에서 접속할 수 있으므로 링크와 비밀번호는 본인만 보관하세요. Hugging Face 토큰을 로그인 비밀번호로 사용하지 마세요.


In [ ]:
from getpass import getpass
import gradio as gr

def extract_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict) and block.get("type") == "text"
        )
    return ""

def respond(message, history):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for entry in history:
        if entry["role"] in ("user", "assistant"):
            messages.append({
                "role": entry["role"],
                "content": extract_text(entry["content"]),
            })
    messages.append({"role": "user", "content": extract_text(message)})
    try:
        with requests.post(
            f"{SERVER_URL}/v1/chat/completions",
            json={
                "model": MODEL_REPO,
                "messages": messages,
                "max_tokens": MAX_NEW_TOKENS,
                "temperature": TEMPERATURE,
                "top_p": TOP_P,
                "top_k": TOP_K,
                "stream": True,
                "chat_template_kwargs": {"enable_thinking": False},
            },
            stream=True, timeout=(10, REQUEST_TIMEOUT),
        ) as response:
            if response.status_code == 400:
                raise gr.Error("요청을 처리할 수 없습니다. 문맥 한도를 넘었다면 대화를 지우고 다시 시작하세요.")
            response.raise_for_status()
            answer = ""
            for line in response.iter_lines(decode_unicode=True):
                if not line or not line.startswith("data: "):
                    continue
                data = line[6:]
                if data == "[DONE]":
                    break
                event = json.loads(data)
                if event.get("error"):
                    raise gr.Error("모델 실행 중 오류가 발생했습니다. 서버 로그를 확인하세요.")
                choices = event.get("choices", [])
                if choices:
                    text = choices[0].get("delta", {}).get("content") or ""
                    if text:
                        answer += text
                        yield answer
            if not answer:
                raise gr.Error("답변이 비어 있습니다. 서버 로그를 확인하세요.")
    except requests.RequestException as error:
        raise gr.Error("모델 서버 연결에 실패했습니다. 모델 실행 셀과 서버 로그를 확인하세요.") from error

if "demo" in globals():
    demo.close()
chat_password = getpass("대화창 로그인 비밀번호: ")
if not chat_password:
    raise ValueError("공유 대화창을 열려면 비밀번호를 입력하세요.")

demo = gr.ChatInterface(
    fn=respond,
    title="Gemma 4 E4B",
    description="한국어로 편하게 대화하세요.",
    textbox=gr.Textbox(placeholder="메시지를 입력하세요", lines=2),
    concurrency_limit=1,
)
demo.queue(max_size=1).launch(
    share=True,
    inline=True,
    server_name=SERVER_HOST,
    server_port=GRADIO_PORT,
    auth=(CHAT_USERNAME, chat_password),
    show_error=False,
)
del chat_password


## 6. 사용 종료
대화를 마치면 이 셀로 공유 링크와 모델 서버를 닫습니다. 이후 **런타임 → 연결 해제 및 런타임 삭제**로 GPU 사용을 종료하세요.


In [ ]:
if "demo" in globals():
    demo.close()
if "server_process" in globals() and server_process.poll() is None:
    server_process.terminate()
    server_process.wait(timeout=30)
print("대화창과 모델 서버 종료 완료")
